# Agent 3: Meal Plan Generator

### Agent Responsibility
> *"Takes macro targets from Agent 2 and generates a full daily meal plan (breakfast, lunch, dinner, snack) with real macro verification using the food database"*

In [9]:
import json
import re
import pandas as pd
from openai import OpenAI


## Setup

In [10]:
HF_TOKEN  = "<YOUR_HF_TOKEN>"
HF_MODEL  = "meta-llama/Llama-3.2-3B-Instruct"

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)

DATA_DIR      = "FoodData_Central_sr_legacy_food_csv_2018-04"
KEY_NUTRIENTS = {1003: "protein", 1004: "fat", 1005: "carbs", 1008: "calories"}
MEAL_SPLITS   = {"breakfast": 0.25, "lunch": 0.35, "dinner": 0.30, "snack": 0.10}


## Food Database
Built from raw USDA data (same source as Agent 4)

In [11]:
def build_food_db():
    foods = pd.read_csv(f"{DATA_DIR}/food.csv")
    food_nutrients = pd.read_csv(f"{DATA_DIR}/food_nutrient.csv")

    food_nutrients["nutrient_id"] = food_nutrients["nutrient_id"].astype(int)
    filtered = food_nutrients[food_nutrients["nutrient_id"].isin(KEY_NUTRIENTS)].copy()
    filtered["nutrient_name"] = filtered["nutrient_id"].map(KEY_NUTRIENTS)

    pivoted = filtered.pivot_table(
        index="fdc_id",
        columns="nutrient_name",
        values="amount",
        aggfunc="first"
    ).reset_index()

    df = pivoted.merge(foods[["fdc_id", "description"]], on="fdc_id")
    df = df.dropna(subset=["calories", "protein", "fat", "carbs"])
    df = df[df["calories"] > 0].reset_index(drop=True)
    df["food_name"] = df["description"].str.lower()
    return df[["food_name", "calories", "protein", "carbs", "fat"]]

print("Loading food database...")
food_df = build_food_db()
print(f"Loaded {len(food_df)} foods")
food_df.head()

Loading food database...
Loaded 7756 foods


,food_name,calories,protein,carbs,fat
0,"pillsbury golden layer buttermilk biscuits, ar...",307.0,5.88,41.18,13.24
1,"pillsbury, cinnamon rolls with icing, refriger...",330.0,4.34,53.42,11.27
2,"kraft foods, shake n bake original recipe, coa...",377.0,6.10,79.80,3.70
3,"george weston bakeries, thomas english muffins",232.0,8.00,46.00,1.80
4,"waffles, buttermilk, frozen, ready-to-heat",273.0,6.58,41.05,9.22


In [12]:
class FoodDatabaseAgent:

    def __init__(self, dataframe):
        self.df = dataframe.copy()
        self.df = self.df.dropna(subset=["calories", "protein", "carbs", "fat"])

    def get_food(self, name):
        name = name.lower().strip()
        # exact substring match first
        matches = self.df[self.df["food_name"].str.contains(name, na=False, regex=False)]
        if not matches.empty:
            return matches.iloc[0].to_dict()
        # try each word
        for word in name.split():
            if len(word) < 3:
                continue
            matches = self.df[self.df["food_name"].str.contains(word, na=False, regex=False)]
            if not matches.empty:
                return matches.iloc[0].to_dict()
        return None

    def calculate_macros(self, name, grams):
        food = self.get_food(name)
        if not food:
            return None
        factor = grams / 100
        return {
            "food_name": food["food_name"],
            "grams": grams,
            "calories": round(food["calories"] * factor, 1),
            "protein":  round(food["protein"]  * factor, 1),
            "carbs":    round(food["carbs"]    * factor, 1),
            "fat":      round(food["fat"]      * factor, 1)
        }

    def calculate_meal(self, items):
        total = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
        details = []
        for name, grams in items:
            macros = self.calculate_macros(name, grams)
            if not macros:
                continue
            details.append(macros)
            total["calories"] += macros["calories"]
            total["protein"]  += macros["protein"]
            total["carbs"]    += macros["carbs"]
            total["fat"]      += macros["fat"]
        total = {k: round(v, 1) for k, v in total.items()}
        return {"items": details, "macros": total}


food_db = FoodDatabaseAgent(food_df)
print("FoodDatabaseAgent ready")

FoodDatabaseAgent ready


## Meal Plan Generator

In [13]:
class MealPlanAgent:

    def __init__(self, food_db: FoodDatabaseAgent, hf_client: Groq):
        self.db = food_db
        self.client = hf_client

    def _call_groq(self, targets: dict, profile: dict) -> dict:
        cal   = targets["daily_calories"]
        prot  = targets["macros"]["protein_g"]
        fat   = targets["macros"]["fat_g"]
        carbs = targets["macros"]["carbs_g"]

        goal      = profile.get("goal", "maintenance")
        diet_type = profile.get("diet_type", "balanced")
        sex       = profile.get("sex", "male")
        weight    = profile.get("weight_kg", 70)

        splits_text = "\n".join(
            f"  - {meal}: {round(cal * pct)} kcal, {round(prot * pct)}g protein, "
            f"{round(fat * pct)}g fat, {round(carbs * pct)}g carbs"
            for meal, pct in MEAL_SPLITS.items()
        )

        example_format = '''{"breakfast": [{"food": "oats", "grams": 80}, {"food": "eggs", "grams": 150}],
  "lunch":     [{"food": "chicken breast", "grams": 200}, {"food": "rice", "grams": 150}],
  "dinner":    [{"food": "salmon", "grams": 180}, {"food": "potato", "grams": 200}],
  "snack":     [{"food": "banana", "grams": 120}, {"food": "peanut butter", "grams": 30}]}'''

        prompt = (
            f"You are a nutrition planner. Create a daily meal plan.\n\n"
            f"User:\n"
            f"- Sex: {sex}\n"
            f"- Weight: {weight} kg\n"
            f"- Goal: {goal}\n"
            f"- Diet: {diet_type}\n\n"
            f"Daily targets: {cal} kcal | {prot}g protein | {fat}g fat | {carbs}g carbs\n\n"
            f"Target per meal:\n{splits_text}\n\n"
            f"Rules:\n"
            f"- Output ONLY valid JSON, no extra text, no markdown\n"
            f"- Use simple common food names (chicken breast, oats, eggs, rice, banana)\n"
            f"- Each meal: 2-3 items with realistic gram amounts\n"
            f"- Respect diet: {diet_type}\n\n"
            f"Output format:\n{example_format}"
        )

        response = self.client.chat.completions.create(
            model=HF_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4
        )

        content = response.choices[0].message.content.strip()
        content = re.sub(r"```json|```", "", content).strip()
        return json.loads(content)

    def _verify_meals(self, meal_json: dict) -> dict:
        verified = {}
        for meal_name, items in meal_json.items():
            pairs = [(item["food"], item["grams"]) for item in items]
            verified[meal_name] = self.db.calculate_meal(pairs)
        return verified

    def _total_macros(self, verified: dict) -> dict:
        total = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
        for meal_data in verified.values():
            for key in total:
                total[key] += meal_data["macros"][key]
        return {k: round(v, 1) for k, v in total.items()}

    def generate(self, agent2_output: dict) -> dict:
        targets = agent2_output["targets"]
        # accept both dataclass and dict for profile
        raw_profile = agent2_output["profile"]
        if hasattr(raw_profile, "__dict__"):
            profile = raw_profile.__dict__
        else:
            profile = raw_profile

        print("Calling Groq to generate meal plan...")
        meal_json = self._call_groq(targets, profile)

        print("Verifying macros with food database...")
        verified = self._verify_meals(meal_json)

        total = self._total_macros(verified)

        return {
            "status": "success",
            "meals": verified,
            "total_macros": total,
            "targets": {**targets["macros"], "daily_calories": targets["daily_calories"]}
        }

## Test

In [14]:
# Simulated Agent 2 output
agent2_output = {
    "status": "success",
    "profile": {
        "age": 25,
        "sex": "male",
        "height_cm": 174,
        "weight_kg": 71,
        "activity_level": "active",
        "goal": "maintenance",
        "diet_type": "balanced",
        "medical_conditions": []
    },
    "targets": {
        "bmr": 1678,
        "tdee": 2894,
        "daily_calories": 2894,
        "macros": {
            "protein_g": 99,
            "fat_g": 80,
            "carbs_g": 443
        }
    }
}

In [15]:
agent3 = MealPlanAgent(food_db, client)
result = agent3.generate(agent2_output)
print(json.dumps(result['total_macros'], indent=2))


## Full Pipeline Function

In [16]:
def run_meal_plan_agent(agent2_output: dict) -> dict:
    if agent2_output.get("status") != "success":
        return {"status": "error", "reason": "Agent 2 output is not successful"}

    agent = MealPlanAgent(food_db, client)
    return agent.generate(agent2_output)

In [17]:
# muscle gain example
muscle_input = {
    "status": "success",
    "profile": {
        "age": 22,
        "sex": "male",
        "height_cm": 180,
        "weight_kg": 80,
        "activity_level": "moderate",
        "goal": "muscle_gain",
        "diet_type": "balanced",
        "medical_conditions": []
    },
    "targets": {
        "bmr": 1877,
        "tdee": 2909,
        "daily_calories": 3209,
        "macros": {
            "protein_g": 160,
            "fat_g": 89,
            "carbs_g": 441
        }
    }
}

output = run_meal_plan_agent(muscle_input)
print(json.dumps(output["total_macros"], indent=2))

Calling Groq to generate meal plan...
Verifying macros with food database...
{
  "calories": 4107.7,
  "protein": 156.0,
  "carbs": 511.8,
  "fat": 164.5
}
